In [1]:
# ====================================================IMPORT=================================================================

from datasets import DatasetDict, load_dataset, load_from_disk, Dataset

import os
import warnings
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    BitsAndBytesConfig,
    set_seed
)

import time
from torch.utils.tensorboard import SummaryWriter

warnings.filterwarnings("ignore")


#from datasets import load_from_disk
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

import pandas as pd
import sys
sys.path.append("D:\\Dissertacao\\slm_tupi")
import os
#from datasets import Dataset
import argparse

from utils.utils_train import pre_process, create_prompt, print_trainable_parameters
from transformers import TrainingArguments, DataCollatorForSeq2Seq

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
import torch
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import logging as transformers_logging
from accelerate import Accelerator

import warnings

warnings.simplefilter("ignore")
transformers_logging.set_verbosity_error()


c:\Users\Rumble\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.5.1+cu121
12.1
True
NVIDIA GeForce RTX 2060


In [2]:
# ========================== CMD Argument Parser ==========================
from types import SimpleNamespace

sys.path.append("D:\\Dissertacao\\slm_tupi\\experiments")
from tupi_exp.config import BASELINE_VAL_SHA256, SPLIT_SEED, check_lora_model, get_experiment
from tupi_exp.splits import ids_hash

experiment = get_experiment("exp01_hparams")
hp = experiment.profile
print(hp)

args = SimpleNamespace(
    per_device_train_batch_size=hp.per_device_train_batch_size,
    per_device_eval_batch_size=hp.per_device_eval_batch_size,
    src_lng="Tupi",
    tgt_lng="Português",
    num_train_epochs=hp.num_train_epochs,
    learning_rate=hp.learning_rate,
    project_root=r"D:\Dissertacao\slm_tupi",
    training_dataset_path = experiment.dataset_path,
    historical_dataset_path = r"D:\Dissertacao\slm_tupi\data\dataset_historico.json",  # human translations (test reference)
    model_path="Qwen/Qwen2.5-3B-Instruct",
    resume_from_checkpoint=False,
    resume_checkpoint_path=None,
    r=hp.r,
    is_peft=True,
    is_unsloth=True,
    is_train_response_only=True,
    seed=hp.seed
)

print("Arguments passed:")
print(f"Train Batch Size: {args.per_device_train_batch_size}")
print(f"Eval Batch Size: {args.per_device_eval_batch_size}")
print(f"Number of Epochs: {args.num_train_epochs}")
print(f"Learning Rate: {args.learning_rate}")
print(f"Project Root: {args.project_root}")
print(f"Training Dataset Path: {args.training_dataset_path}")
print(f"Historical Dataset Path: {args.historical_dataset_path}")
print(f"Model path: {args.model_path}")
print(f"tgt_lng: {args.tgt_lng}")
print(f"src_lng: {args.src_lng}")
print(f"Resume from checkpoint: {args.resume_from_checkpoint}")
print(f"Resume checkpoint path: {args.resume_checkpoint_path}")
print(f"r: {args.r}")
print(f"is_peft: {args.is_peft}")
print(f"is_unsloth: {args.is_unsloth}")
print(f"is_train_response_only: {args.is_train_response_only}")
print(f"Seed: {args.seed}")

Seed 99: primeiro uso, gerando o split e o v3 em splits/ e data/
Length of inputs before: 1975
Length of inputs after: 1975
Removed rows: 0
HParams(name='hp_v2', r=64, lora_alpha=64, lora_dropout=0.05, learning_rate=0.0001, lr_scheduler_type='cosine', warmup_ratio=0.05, weight_decay=0.01, max_grad_norm=1.0, num_train_epochs=3, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, target_modules=('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'), seed=99, evaluation_strategy='steps', eval_steps=1000, logging_steps=1000, save_strategy='epoch')
Arguments passed:
Train Batch Size: 1
Eval Batch Size: 1
Number of Epochs: 3
Learning Rate: 0.0001
Project Root: D:\Dissertacao\slm_tupi
Training Dataset Path: D:\Dissertacao\slm_tupi\data\dataset_sintetico_aumentado.json
Historical Dataset Path: D:\Dissertacao\slm_tupi\data\dataset_historico.json
Model path: Qwen/Qwen2.5-3B-Instruct
tgt_lng: Português
src_lng: Tupi
Resume from checkp

In [3]:
learning_rate = args.learning_rate # Learning rate for the optimizer
per_device_train_batch_size = args.per_device_train_batch_size  # Batch size for training per device
per_device_eval_batch_size = args.per_device_eval_batch_size  # Batch size for evaluation per device
num_train_epochs = args.num_train_epochs  # Number of epochs for training
training_dataset_path = args.training_dataset_path
historical_dataset_path = args.historical_dataset_path
project_root = args.project_root
model_path = args.model_path
resume_from_checkpoint = args.resume_from_checkpoint
resume_checkpoint_path = args.resume_checkpoint_path
src_lng = args.src_lng
tgt_lng = args.tgt_lng
r = args.r
is_peft = args.is_peft
is_unsloth = args.is_unsloth
is_train_response_only = args.is_train_response_only
train_seed = args.seed

if is_unsloth:
    #from unsloth import is_bfloat16_supported
    #from unsloth.chat_templates import train_on_responses_only
    #from unsloth import FastLanguageModel
    #fp16 = not is_bfloat16_supported()
    #bf16 = is_bfloat16_supported(),
    fp16 = False
    bf16 = False
else:
    fp16 = False
    bf16 = False



#model_name = model_path.split("/")[-1]
train_ratio = 1.0  # Number of samples to be used for training and evaluation
warmup_ratio = hp.warmup_ratio
logging_steps = hp.logging_steps
evaluation_strategy=hp.evaluation_strategy  # eval loss on the 30% test split: monitoring only, never used to pick checkpoints or epochs
save_strategy=hp.save_strategy
eval_steps=hp.eval_steps
max_grad_norm = hp.max_grad_norm
MAX_LEN = experiment.max_seq_length
weight_decay = hp.weight_decay
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.
train_seed = train_seed

if is_peft:
    target_modules = list(hp.target_modules)
    lora_alpha = hp.lora_alpha
    lora_dropout = hp.lora_dropout
    random_state = train_seed

current = time.time()
formatted_time = time.strftime("%m_%d_%H_%M", time.localtime(current))

if resume_from_checkpoint:
    output_dir = resume_checkpoint_path

else:

    input_file_name = os.path.splitext(
        os.path.basename(training_dataset_path)
    )[0]

    if is_peft:

        output_dir = (
            f"logs/{experiment.name}/peft_{r}_{src_lng[:2]}_{tgt_lng[:2]}/"
            f"fit_{formatted_time}_{train_ratio}_{input_file_name}-SEED{train_seed}"
        )

    else:

        output_dir = (
            f"logs/full_{src_lng[:2]}_{tgt_lng[:2]}/"
            f"fit_{formatted_time}_{train_ratio}_{input_file_name}-SEED{train_seed}"
        )

if resume_from_checkpoint and resume_checkpoint_path is None:
    raise ValueError("Please provide a checkpoint path to resume training from")

In [4]:
# ========================== dataset preparation ==========================

#train_dataset_path = os.path.abspath(os.path.join(project_root, training_dataset_path))
train_dataset_path = os.path.abspath(os.path.join(training_dataset_path))
sys.path.append(project_root)

train_dataset_df = pd.read_json(train_dataset_path) #, lines=True)
pre_processed_dataset_df = pre_process(train_dataset_df)

if not isinstance(pre_processed_dataset_df, pd.DataFrame):
    raise TypeError("data_preprocess should return a pandas DataFrame.")

dataset = Dataset.from_pandas(pre_processed_dataset_df)

dataset = dataset.shuffle(seed=SPLIT_SEED)
dataset = dataset.map(
    lambda x, idx: {
        "split": "train" if idx < int(0.7 * len(dataset)) else "test"
    },
    with_indices=True
)

# Filter by split
train_dataset = dataset.filter(lambda x: x["split"] == "train")
test_dataset = dataset.filter(lambda x: x["split"] == "test")

# Same validation as the baseline in every experiment (P6)
assert ids_hash(test_dataset["id"]) == BASELINE_VAL_SHA256, "validation differs from the baseline"

# Select subset
train_dataset = train_dataset.select(range(int(len(train_dataset) * train_ratio)))
test_dataset = test_dataset.select(range(int(len(test_dataset) * train_ratio)))  # Avoid out-of-range error

# Test reference: human translation for historical sentences (their training label is Gemini's), template label for synthetic ones
historical_df = pd.read_json(historical_dataset_path)
gold_by_id = dict(zip(historical_df["id"], historical_df["output"]))
test_dataset = test_dataset.map(lambda x: {"Referência": gold_by_id.get(x["id"], x["output"])})

# Rename columns
train_dataset = train_dataset.rename_columns({
    "input": "Tupi",
    "output": "Português",
})

test_dataset = test_dataset.rename_columns({
    "input": "Tupi",
    "output": "Português",
})

# Save the split next to the checkpoints, so the evaluation reads the test set of this same run
os.makedirs(output_dir, exist_ok=True)
test_dataset.to_json(os.path.join(output_dir, "test_dataset.json"))
train_dataset.to_json(os.path.join(output_dir, "train_dataset.json"))

# Keep Qwen's own pad token (<|endoftext|>): padding with eos (<|im_end|>) masks the end of the answer out of the loss
tokenizer = AutoTokenizer.from_pretrained(model_path)

# eos_rep=0: the chat template already closes the answer with <|im_end|>
train_dataset = train_dataset.map(
    lambda sample: {
        "full_prompt": create_prompt(sample, src_lng=src_lng, tgt_lng=tgt_lng, eos_rep=0, mode="train", tokenizer=tokenizer)["full_prompt"]
    }
).select_columns(["full_prompt"])

test_dataset = test_dataset.map(
    lambda sample: {
        "full_prompt": create_prompt(sample, src_lng=src_lng, tgt_lng=tgt_lng, eos_rep=0, mode="train", tokenizer=tokenizer)["full_prompt"]
    }
).select_columns(["full_prompt"])

print (train_dataset["full_prompt"][0])

def tokenize_function(examples):
    return tokenizer(
        examples["full_prompt"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["full_prompt"])
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=["full_prompt"])

print("Dataset tokenized:")
print(tokenized_train_dataset[0])

Length of inputs before: 1975
Length of inputs after: 1975
Removed rows: 0


Map: 100%|██████████| 593/593 [00:00<00:00, 3443.24 examples/s]


<|im_start|>system
Você é um assistente de IA muito útil para traduções.<|im_end|>
<|im_start|>user
Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

i tyb itãetá ybyku'i resé<|im_end|>
<|im_start|>assistant
há muitas pedras sobre a areia<|im_end|>



Map: 100%|██████████| 593/593 [00:00<00:00, 12847.16 examples/s]

Dataset tokenized:
{'input_ids': [151644, 8948, 198, 69286, 3958, 4443, 7789, 6817, 409, 43090, 33750, 137741, 3348, 4685, 84, 15249, 13, 151645, 198, 151644, 872, 198, 42834, 84, 4360, 297, 93072, 68, 32025, 976, 96097, 72, 3348, 22234, 84, 36830, 13, 57649, 18409, 4284, 64066, 80367, 2782, 5908, 136836, 24870, 3449, 4942, 382, 72, 13580, 65, 432, 3202, 295, 1953, 379, 1694, 12133, 62550, 592, 963, 151645, 198, 151644, 77091, 198, 71, 1953, 53760, 300, 10477, 12784, 15131, 264, 525, 685, 151645, 198, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [5]:
# Seed before the LoRA matrices are initialised (the Trainer only seeds itself later, when it is created)
set_seed(train_seed)

if is_unsloth:
    bnb_config = BitsAndBytesConfig(
    load_in_4bit=load_in_4bit,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
    model_path, #model_name,
    trust_remote_code=True
    )

    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
    model_path, #model_name,
    quantization_config=bnb_config if load_in_4bit else None,
    #device_map="auto",
    trust_remote_code=True,
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    if load_in_4bit:
        model = prepare_model_for_kbit_training(model)

    if is_peft:
        lora_config = LoraConfig(
            r=r,
            target_modules=target_modules,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
            task_type="CAUSAL_LM",  # without it the Trainer finds no "labels" argument and never computes eval_loss
            use_rslora=False,
            #random_state = random_state,
            #loftq_config = None,
        )

        model = get_peft_model(model, lora_config)
# Using transformer huggingface   
else:
    model = AutoModelForCausalLM.from_pretrained(model_path)
    model.config.use_cache = False
    if is_peft:
        lora_config = LoraConfig(
            r=r, 
            target_modules=target_modules, 
            lora_alpha=lora_alpha, 
            lora_dropout=lora_dropout, 
            bias="none", 
            task_type="CAUSAL_LM",
            use_rslora=False,
            loftq_config=None  # And without LoftQ
        )
        model = get_peft_model(model, lora_config)

print (model)
print(print_trainable_parameters(model))
check_lora_model(model, hp)

Loading checkpoint shards: 100%|██████████| 2/2 [00:39<00:00, 19.79s/it]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Lin

In [6]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Layer: {name}, Shape: {param.shape}, Trainable parameters: {param.numel()}")
    else:
        print(f"Layer: {name}, Shape: {param.shape}, Non-trainable parameters: {param.numel()}")

Layer: base_model.model.model.embed_tokens.weight, Shape: torch.Size([151936, 2048]), Non-trainable parameters: 311164928
Layer: base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight, Shape: torch.Size([2048, 2048]), Non-trainable parameters: 4194304
Layer: base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias, Shape: torch.Size([2048]), Non-trainable parameters: 2048
Layer: base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight, Shape: torch.Size([64, 2048]), Trainable parameters: 131072
Layer: base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight, Shape: torch.Size([2048, 64]), Trainable parameters: 131072
Layer: base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight, Shape: torch.Size([256, 2048]), Non-trainable parameters: 524288
Layer: base_model.model.model.layers.0.self_attn.k_proj.base_layer.bias, Shape: torch.Size([256]), Non-trainable parameters: 256
Layer: base_model.model.model.layers.0.self_attn.k_proj.lo

In [7]:
def train_ddp_accelerate_sft():
    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        gradient_accumulation_steps=hp.gradient_accumulation_steps,
        warmup_ratio=warmup_ratio,
        evaluation_strategy=evaluation_strategy,
        save_strategy=save_strategy,
        logging_steps=logging_steps,
        eval_steps=eval_steps,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        fp16 = fp16,
        bf16 =bf16,
        max_grad_norm=max_grad_norm,
        group_by_length=True,
        lr_scheduler_type=hp.lr_scheduler_type,
        report_to="tensorboard",
        remove_unused_columns=False,
        disable_tqdm=False,
        seed = train_seed,
        ddp_find_unused_parameters=False, 
        dataloader_num_workers=2,
        dataset_text_field = "full_prompt",
        max_seq_length = MAX_LEN,
        dataset_num_proc = 2,
        packing = False, # Can make training 5x faster for short sequences.
        # load_best_model_at_end=True,
    )

    # Loss only on the answer: the fixed prompt and the Tupi source are masked out (-100)
    data_collator = DataCollatorForCompletionOnlyLM(
        response_template="<|im_start|>assistant\n",
        tokenizer=tokenizer,
    ) if is_train_response_only else None

    trainer = SFTTrainer(
        model=model,
        #train_dataset=train_dataset,
        #eval_dataset=test_dataset,
        train_dataset = tokenized_train_dataset,
        eval_dataset = tokenized_test_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        args=training_args
    )

    trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    print("Finished training SFT.")
    return trainer_stats

In [8]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 2060. Max memory = 6.0 GB.
0.0 GB of memory reserved.


In [9]:
print(train_dataset.column_names)

print(type(train_dataset[2]["full_prompt"]))
print(train_dataset[2]["full_prompt"][:400])

['full_prompt']
<class 'str'>
<|im_start|>system
Você é um assistente de IA muito útil para traduções.<|im_end|>
<|im_start|>user
Traduza o seguinte texto em Tupi para Português. Não inclua informações adicionais ou conteúdo irrelevante.

xe rapixara oîkó ka'ape<|im_end|>
<|im_start|>assistant
o meu semelhante está na mata<|im_end|>



In [ ]:
trainer_stats = None

def main():
    trainer_stats = train_ddp_accelerate_sft()
    return trainer_stats

if __name__ == "__main__":
    trainer_stats = main()

 11%|█         | 56/516 [1:23:03<11:26:06, 89.49s/it]

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

52505.7261 seconds used for training.
875.1 minutes used for training.
Peak reserved memory = 14.971 GB.
Peak reserved memory for training = 14.971 GB.
Peak reserved memory % of max memory = 249.517 %.
Peak reserved memory for training % of max memory = 249.517 %.
